# 32 · Compaction: when history outgrows the window
Level L3 · Part 4 · professional · ~35 min · builds on 05, 31

## Goal
Hit a real context-overflow error from Gemini, then keep a 60-turn Scout chat under an 8,000-token
budget by clearing old tool results and summarising old turns, and measure with a context rot test that
the compacted chat answers the same questions as the full one.

## The idea
Lesson 05: the window is finite, and the APIs refuse a request that does not fit. Lesson 31: trim old tool
results. That slows the growth, but every turn still adds a question and an answer, so a long enough chat
overflows anyway. **Compaction** is the next step: when the history passes a **trigger**, replace the older
turns with a short **summary** and keep the latest turns word for word. The chat now has a ceiling.
The risk: a summary can silently drop a fact the user needs later. So you test it with a **context rot test**:
plant facts early, ask about them late, and score the answers against the tokens sent.

Analogy: minutes of a long meeting. Nobody rereads the recording; they read the minutes and the last few slides.

## Picture
```mermaid
flowchart LR
    H["history<br/>turns 1..60"] --> T{"over the<br/>trigger?"}
    T -- no --> R["send as is"]
    T -- yes --> O["old turns 1..54"] --> S["summarise<br/>(model call)"]
    S --> C["summary + turns 55..60<br/>word for word"]
    C --> R
    R --> M[model] --> P["context rot test<br/>score vs tokens"]
```

## Step by step
### 1. Start the recorder and set up
The recorder first, as in lesson 31. Without a key the Gemini calls replay from
`../data/llm_cassettes/32_compaction.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("32_compaction")

LLM mode: replay (32_compaction.yaml)


Setup from lesson 31, unchanged: the client, the pinned model, and the tiktoken counter
(an estimate; Gemini's own count differs a little, lesson 12).

In [2]:
import json, os, random, re
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

os.environ["TIKTOKEN_CACHE_DIR"] = "../data/tiktoken_cache"
import tiktoken
ENC = tiktoken.get_encoding("o200k_base")

def tokens(text: str) -> int:
    return len(ENC.encode(text))

### 2. Scout's rover log
As in lesson 31, Scout's notes are a rover's daily log, generated with a fixed seed. This time we keep
300 sols (Martian days) as a list, and a `read_log` tool returns a range of them.

In [3]:
rng = random.Random(32)
SOLS = [{"sol": s, "drove": rng.randint(0, 120), "wind": rng.randint(1, 15),
         "opacity": round(rng.uniform(0.2, 1.5), 2), "battery": rng.randint(40, 100)}
        for s in range(1, 301)]
LOG = [f"Sol {d['sol']}: drove {d['drove']} m, wind {d['wind']} m/s, dust opacity {d['opacity']:.2f}, "
       f"battery {d['battery']}%." for d in SOLS]

def read_log(first_sol: int, last_sol: int) -> str:
    """Return the log lines for sols first_sol..last_sol."""
    return "\n".join(LOG[first_sol - 1:last_sol])

print(read_log(1, 2))
print(tokens(LOG[0] + "\n"), "tokens per log line (tiktoken estimate)")

Sol 1: drove 9 m, wind 15 m/s, dust opacity 0.48, battery 59%.
Sol 2: drove 89 m, wind 15 m/s, dust opacity 0.51, battery 41%.
26 tokens per log line (tiktoken estimate)


### 3. A real overflow
How big is Gemini's window? Ask it. We send 40,000 log lines plus a question. It costs nothing: the request
is rejected before the model runs. (The recording stores the error, not the 2.5 MB of text we sent.)

In [4]:
huge = (LOG[0] + "\n") * 40_000
print(f"about {tokens(LOG[0] + chr(10)) * 40_000:,} tokens by our counter, {len(huge):,} characters")

about 1,040,000 tokens by our counter, 2,520,000 characters


Now send it, and catch the SDK's `ClientError` (lesson 13) to read the code and message.

In [5]:
try:
    client.models.generate_content(model=MODEL, contents=huge + "How many sols are in this log?",
                                   config=types.GenerateContentConfig(max_output_tokens=20,
                                                                      automatic_function_calling=NO_AFC))
except errors.ClientError as err:
    print("error", err.code, err.status)
    print(err.message)

error 400 INVALID_ARGUMENT
The input token count exceeds the maximum number of tokens allowed 1048576.


A 400 `INVALID_ARGUMENT`: the input is over 1,048,576 tokens, this model's window. No partial answer, no
trimming: the API refuses and leaves the choice to you, as lesson 05 said.

Scout will not wait for a million tokens. Every token is paid on every turn (lesson 12) and recall gets
worse as the window fills (context rot, lesson 05). So Scout gets its own **budget**: 8,000 tokens per
request, checked before sending, with the same kind of error.

In [6]:
WINDOW = 8_000                    # Scout's budget per request, far below Gemini's 1,048,576

class ContextWindowExceeded(Exception):
    pass

SYSTEM = ("You are Scout, a research assistant over the user's rover log. "
          "Answer only from this chat and the tool results in it.")

def context_tokens(messages: list[dict]) -> int:
    body = lambda m: m.get("text") or m.get("content") or json.dumps(m.get("tool_calls", []))
    return tokens(SYSTEM) + sum(4 + tokens(body(m)) for m in messages)

def check_budget(messages: list[dict]) -> int:
    n = context_tokens(messages)
    if n > WINDOW:
        raise ContextWindowExceeded(f"prompt is too long: {n:,} tokens > {WINDOW:,} maximum")
    return n

### 4. A 60-turn chat
The chat is scripted so every run is the same. Turns 1 to 4 plant facts the user will ask about later:
the project, the date, a rule, a number. Turns 5 to 60 each ask about five sols: Scout calls `read_log`
and answers from the result. Messages use lesson 31's format, plus a `turn` number.

In [7]:
FACT_TURNS = [
    ("Hi Scout. My project is a 10-minute talk on dust storms on Mars, for my astronomy club.",
     "Great topic. I'll help you dig through the rover log."),
    ("The talk is on 14 November, so we have five weeks.", "Noted: the talk is on 14 November."),
    ("Please use metric units only. My club voted against miles.", "Understood: metric units only."),
    ("A sol is 24 hours and 39 minutes. How many minutes is that?", "A sol is 1,479 minutes (24 x 60 + 39)."),
]

def block_answer(a: int, b: int) -> str:
    block = SOLS[a - 1:b]
    dusty = max(block, key=lambda d: d["opacity"])
    return (f"Sols {a}-{b}: the rover drove {sum(d['drove'] for d in block)} m in total; "
            f"the dustiest was sol {dusty['sol']} (opacity {dusty['opacity']:.2f}).")

`turn_messages` builds the messages of one turn. Tool turns have four: question, tool call, tool result,
answer.

In [8]:
def turn_messages(t: int) -> list[dict]:
    if t <= len(FACT_TURNS):
        question, answer = FACT_TURNS[t - 1]
        return [{"role": "user", "text": question, "turn": t}, {"role": "assistant", "text": answer, "turn": t}]
    a = (t - 5) * 5 + 1
    b = a + 4
    call = {"id": f"call_{t}", "name": "read_log", "args": {"first_sol": a, "last_sol": b}}
    return [{"role": "user", "text": f"What happened on sols {a} to {b}?", "turn": t},
            {"role": "assistant", "tool_calls": [call], "turn": t},
            {"role": "tool", "call_id": call["id"], "name": "read_log", "args": call["args"],
             "content": read_log(a, b), "turn": t},
            {"role": "assistant", "text": block_answer(a, b), "turn": t}]

history = [m for t in range(1, 61) for m in turn_messages(t)]
for t in (10, 20, 30, 40, 50, 60):
    print(f"after turn {t}: {context_tokens([m for m in history if m['turn'] <= t]):6,} tokens")

after turn 10:  1,489 tokens
after turn 20:  3,689 tokens
after turn 30:  5,889 tokens
after turn 40:  8,089 tokens
after turn 50: 10,289 tokens
after turn 60: 12,489 tokens


Now turn 61 sends the whole history, as every turn does (lesson 08). Our budget check refuses it.

In [9]:
try:
    check_budget(history)
except ContextWindowExceeded as err:
    print("error:", err)
first_over = next(t for t in range(1, 61) if context_tokens([m for m in history if m["turn"] <= t]) > WINDOW)
print("first turn over the budget:", first_over)

error: prompt is too long: 12,489 tokens > 8,000 maximum


first turn over the budget: 40


### 5. Clear old tool results first
Lesson 31's cheapest fix: keep every message, but replace the content of tool results older than the last
`keep_turns` turns with a short note. The history itself is not edited; we build a view.

In [10]:
def clear_tool_results(messages: list[dict], keep_turns: int = 2) -> list[dict]:
    last = max(m["turn"] for m in messages)
    return [{**m, "content": f"[read_log{tuple(m['args'].values())} cleared]"}
            if m["role"] == "tool" and m["turn"] <= last - keep_turns else m for m in messages]

cleared = clear_tool_results(history)
print(f"full: {context_tokens(history):,} tokens -> cleared: {context_tokens(cleared):,} tokens")
grow = context_tokens(clear_tool_results([m for m in history if m["turn"] <= 60])) - \
       context_tokens(clear_tool_results([m for m in history if m["turn"] <= 59]))
print("each new turn still adds about", grow, "tokens")
print("turns until 8,000 again: about", (WINDOW - context_tokens(cleared)) // grow)

full: 12,489 tokens -> cleared: 6,063 tokens
each new turn still adds about 101 tokens
turns until 8,000 again: about 19


Clearing halves the chat (6,063 tokens), but every new question, call and answer still adds about 100
tokens, so it passes 8,000 again about 19 turns later. Clearing delays the overflow;
it does not stop it. For a ceiling we must shrink the old turns themselves.

### 6. Compaction by hand
`compact` splits the history at a turn boundary: the last `keep_turns` turns stay word for word, everything
older goes to a `summarise` function, and its text becomes one `summary` message at the front. Splitting by
whole turns keeps each tool call next to its result (lesson 22).

In [11]:
def compact(messages: list[dict], summarise, keep_turns: int = 6) -> list[dict]:
    turns = sorted({m["turn"] for m in messages if m["role"] != "summary"})
    keep_from = turns[-keep_turns]
    old = [m for m in messages if m["role"] == "summary" or m["turn"] < keep_from]
    recent = [m for m in messages if m["role"] != "summary" and m["turn"] >= keep_from]
    return [{"role": "summary", "text": summarise(old), "turn": keep_from - 1}] + recent

First a summariser with no model, so we can see the mechanics. `extract_summary` keeps any earlier summary,
the user's own statements (not their questions) and Scout's final answers. It drops questions, tool calls
and tool results.

In [12]:
def extract_summary(messages: list[dict]) -> str:
    lines = []
    for m in messages:
        if m["role"] == "summary":
            lines.append(m["text"])
        elif m["role"] == "user" and not m["text"].endswith("?"):
            lines.append("User said: " + m["text"])
        elif m["role"] == "assistant" and "text" in m:
            lines.append("Scout said: " + m["text"])
    return "\n".join(lines)

compacted = compact(history, extract_summary)
print(f"full: {context_tokens(history):,} tokens -> compacted: {context_tokens(compacted):,} tokens")
print(compacted[0]["text"][:400])

full: 12,489 tokens -> compacted: 3,115 tokens
User said: Hi Scout. My project is a 10-minute talk on dust storms on Mars, for my astronomy club.
Scout said: Great topic. I'll help you dig through the rover log.
User said: The talk is on 14 November, so we have five weeks.
Scout said: Noted: the talk is on 14 November.
User said: Please use metric units only. My club voted against miles.
Scout said: Understood: metric units only.
Scout said: A


### 7. A summary written by the model
`extract_summary` still keeps one line per turn, so it grows too. A model can write a real summary.
The prompt says what must survive: the user's goal, dates and rules word for word, and every number.
`transcript` turns messages into plain text for the summariser.

In [13]:
def transcript(messages: list[dict]) -> str:
    out = []
    for m in messages:
        if m["role"] == "summary":
            out.append("EARLIER SUMMARY:\n" + m["text"])
        elif m["role"] == "tool":
            out.append(f"TOOL read_log{tuple(m['args'].values())}:\n{m['content']}")
        elif "text" in m:
            out.append(f"{m['role'].upper()}: {m['text']}")
    return "\n".join(out)

SUMMARY_PROMPT = """Summarise this chat between a user and Scout so Scout can continue it without the chat.
Keep, word for word: the user's goal, every date, and every rule or preference they gave.
Keep every number Scout gave in an answer, one short line per block of sols (total metres, dustiest sol).
Leave out greetings and the raw log lines. Plain bullet points only.

CHAT:
"""

The Gemini summariser: one call, temperature 0. If you run without a recording or key, the cell is skipped
and `summarise_with_gemini` falls back to `extract_summary`, so later cells still run.

In [14]:
summary_log = []                              # every summary the model wrote, for inspection

def summarise_with_gemini(messages: list[dict], prompt: str = SUMMARY_PROMPT) -> str:
    try:
        resp = client.models.generate_content(
            model=MODEL, contents=prompt + transcript(messages),
            config=types.GenerateContentConfig(temperature=0, max_output_tokens=2000,
                                               automatic_function_calling=NO_AFC))
        summary_log.append({"in": resp.usage_metadata.prompt_token_count, "text": resp.text})
        return resp.text
    except Exception as err:                  # no key and no recording: fall back, say so
        print("summariser fallback:", type(err).__name__)
        return extract_summary(messages)

Compact the 60-turn history with it, keeping the last 6 turns, and look at the start and end of the summary.

In [15]:
compacted = compact(history, summarise_with_gemini)
print(f"summariser read {summary_log[-1]['in']:,} tokens (Gemini's count)")
print(f"full: {context_tokens(history):,} tokens -> compacted: {context_tokens(compacted):,} tokens")
print(compacted[0]["text"][:450], "\n...\n" + compacted[0]["text"][-160:])

summariser read 12,154 tokens (Gemini's count)


full: 12,489 tokens -> compacted: 2,488 tokens
* User's goal: A 10-minute talk on dust storms on Mars, for my astronomy club.
* 14 November
* Please use metric units only. My club voted against miles.
* Sols 1-5: 352 m, sol 5 (opacity 0.81)
* Sols 6-10: 203 m, sol 8 (opacity 1.29)
* Sols 11-15: 178 m, sol 14 (opacity 1.26)
* Sols 16-20: 378 m, sol 18 (opacity 1.48)
* Sols 21-25: 243 m, sol 25 (opacity 1.44)
* Sols 26-30: 193 m, sol 26 (opacity 1.32)
* Sols 31-35: 299 m, sol 33 (opacity 1.31)
 
...
sol 234 (opacity 1.36)
* Sols 236-240: 213 m, sol 237 (opacity 1.43)
* Sols 241-245: 240 m, sol 241 (opacity 1.42)
* Sols 246-250: 359 m, sol 250 (opacity 1.43)


### 8. Compaction inside the chat
In a real chat you compact on the way, not once at the end. `chat_with_compaction` replays the 60 turns.
Before each request it checks a **trigger**, 6,000 tokens (75% of the budget), and compacts when the
history is over it. It records the tokens sent on every request. We use the offline summariser here.

In [16]:
TRIGGER = 6_000

def chat_with_compaction(n_turns: int, summarise, keep_turns: int = 6) -> dict:
    live, sizes, compactions = [], [], 0
    for t in range(1, n_turns + 1):
        live += turn_messages(t)
        if context_tokens(live) > TRIGGER:
            live = compact(live, summarise, keep_turns)
            compactions += 1
        sizes.append(check_budget(live))          # raises if we ever go over 8,000
    return {"live": live, "sizes": sizes, "compactions": compactions}

run = chat_with_compaction(60, extract_summary)
print("compactions:", run["compactions"], "| largest request:", max(run["sizes"]), "tokens")
print("tokens sent at turns 10, 20, ... 60:", run["sizes"][9::10])

compactions: 2 | largest request: 5898 tokens
tokens sent at turns 10, 20, ... 60: [1489, 3689, 5889, 4138, 2972, 5172]


Sixty turns, never over 8,000 tokens. The size saws up and down: it grows to the trigger, drops at a
compaction, grows again. Each compaction folds the previous summary into the new one ("compact again").

### 9. The context rot test
Smaller is useless if Scout forgets. We plant-and-probe: five questions whose answers sit in turns 1 to 4
and in turn 11 (sols 31 to 35), all far from the end. Each answer is scored by a keyword.

In [17]:
dusty_31 = max(SOLS[30:35], key=lambda d: d["opacity"])["sol"]
PROBES = [("What is my project about?", ["dust storm"]),
          ("When is my talk?", ["14 november"]),
          ("Which units must the talk use?", ["metric"]),
          ("How many minutes is a sol?", ["1479", "1,479"]),
          ("Which sol was the dustiest from sol 31 to sol 35?", [f"sol {dusty_31}"])]

def score(answer_text: str) -> list[bool]:
    lines = {int(n): a.lower() for n, a in re.findall(r"^\s*(\d)[.)]\s*(.+)$", answer_text, flags=re.M)}
    return [any(k in lines.get(i, "") for k in keys) and "not in chat" not in lines.get(i, "")
            for i, (_, keys) in enumerate(PROBES, start=1)]

print("expected dustiest sol:", dusty_31)
print(score(f"1. A talk on dust storms\n2. 14 November\n3. miles\n4. 1,479\n5. Sol {dusty_31}"))

expected dustiest sol: 33
[True, True, False, True, True]


Three versions of the history go to Gemini with the same five questions in one message:
**full** (over our budget, but inside Gemini's window), **sliding window** (lesson 05: newest whole turns
that fit in 6,000 tokens) and **compacted** (step 7). `to_contents` maps our messages to Gemini turns; a
summary goes into the system instruction.

In [18]:
def sliding_window(messages: list[dict], budget: int = TRIGGER) -> list[dict]:
    turns = sorted({m["turn"] for m in messages}, reverse=True)
    keep = [t for i, t in enumerate(turns)
            if context_tokens([m for m in messages if m["turn"] >= t]) <= budget]
    return [m for m in messages if m["turn"] >= min(keep)]

def to_contents(messages: list[dict]) -> tuple[str, list[types.Content]]:
    system, contents = SYSTEM, []
    for m in messages:
        if m["role"] == "summary":
            system += "\n\nSummary of the earlier part of this chat:\n" + m["text"]
        elif m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "tool":                # tool exchange shown as text in Scout's turn
            contents.append(types.Content(role="model", parts=[types.Part(
                text=f"[read_log{tuple(m['args'].values())} returned]\n{m['content']}")]))
        elif "text" in m:
            contents.append(types.Content(role="model", parts=[types.Part(text=m["text"])]))
    return system, contents

Our scripted history was not written by Gemini, so tool calls become plain text in Scout's turns rather
than real function-call parts. The facts are the same. Now ask, and score.

In [19]:
QUESTIONS = ("Answer from this chat only. If the chat does not say, write 'not in chat'. "
             "Reply with 5 numbered lines.\n" + "\n".join(f"{i}. {q}" for i, (q, _) in enumerate(PROBES, 1)))

def ask_probes(messages: list[dict]) -> dict:
    system, contents = to_contents(messages)
    contents.append(types.Content(role="user", parts=[types.Part(text=QUESTIONS)]))
    resp = client.models.generate_content(model=MODEL, contents=contents, config=types.GenerateContentConfig(
        system_instruction=system, temperature=0, max_output_tokens=300, automatic_function_calling=NO_AFC))
    return {"tokens": resp.usage_metadata.prompt_token_count, "text": resp.text, "score": score(resp.text)}

Ask the three versions and print each score next to its prompt tokens, then the raw answers.

In [20]:
results = {}
for name, view in [("full", history), ("sliding window", sliding_window(history)), ("compacted", compacted)]:
    results[name] = ask_probes(view)
    r = results[name]
    print(f"{name:15} {r['tokens']:6,} prompt tokens | {sum(r['score'])}/5 correct | {r['score']}")
for name, r in results.items():
    print(f"\n{name}:\n{r['text'].strip()}")

full            13,226 prompt tokens | 5/5 correct | [True, True, True, True, True]
sliding window   6,507 prompt tokens | 0/5 correct | [False, False, False, False, False]
compacted        3,131 prompt tokens | 4/5 correct | [True, True, True, False, True]

full:
1. dust storms on Mars
2. 14 November
3. metric units only
4. 1,479 minutes
5. sol 33

sliding window:
1. not in chat
2. not in chat
3. not in chat
4. not in chat
5. not in chat

compacted:
1. A 10-minute talk on dust storms on Mars, for my astronomy club.
2. 14 November
3. metric units only
4. not in chat
5. sol 33


Read the scores against the tokens (Gemini's own prompt-token counts from `usage_metadata`).
In the recorded run: **full**, 13,226 tokens, 5/5. **Sliding window**, 6,507 tokens, 0/5: it kept only
the newest turns and every planted fact was older (Gemini said "not in chat" rather than guess).
**Compacted**, 3,131 tokens, 4/5: under a quarter of the tokens, but the summary dropped "a sol is 1,479
minutes". Our prompt asked for one line per block of sols, and that fact was not a block. Nothing
raised an error; only the test showed the loss.

### 10. Fix the summary prompt, test again
The fix goes in the prompt: ask for every other fact Scout stated, not just the per-block lines. Then
rerun the same probes. Changing the prompt without rerunning the test is how facts get lost in production.

In [21]:
SUMMARY_PROMPT_V2 = SUMMARY_PROMPT.replace(
    "Leave out greetings", "Also keep every other fact or number Scout stated, such as unit conversions.\n"
    "Leave out greetings")
print(SUMMARY_PROMPT_V2)

Summarise this chat between a user and Scout so Scout can continue it without the chat.
Keep, word for word: the user's goal, every date, and every rule or preference they gave.
Keep every number Scout gave in an answer, one short line per block of sols (total metres, dustiest sol).
Also keep every other fact or number Scout stated, such as unit conversions.
Leave out greetings and the raw log lines. Plain bullet points only.

CHAT:



Compact again with prompt v2 and ask the same five probes.

In [22]:
compacted_v2 = compact(history, lambda old: summarise_with_gemini(old, SUMMARY_PROMPT_V2))
results["compacted, prompt v2"] = r = ask_probes(compacted_v2)
print(f"compacted v2: {context_tokens(compacted_v2):,} tokens by our counter")
print(f"{r['tokens']:,} prompt tokens | {sum(r['score'])}/5 correct | {r['score']}")
print(r["text"].strip())

compacted v2: 2,914 tokens by our counter
3,546 prompt tokens | 5/5 correct | [True, True, True, True, True]
1. Dust storms on Mars
2. 14 November
3. Metric units only
4. 1,479 minutes
5. Sol 33 (opacity 1.31)


In the recorded run prompt v2 scores 5/5 on 3,546 prompt tokens, about a quarter of the full chat's
13,226: the same answers for a quarter of the tokens. Five probes in one run is a small test; lesson 48
builds a proper eval set.

### 11. The vendor versions
You just built what these features do. From the vendors' docs (RESEARCH.md section 41):

| Vendor | Feature | How it works |
|---|---|---|
| Anthropic | Compaction at a token threshold (beta `compact-2026-01-12`) | `context_management.edits=[{"type": "compact_20260112"}]`; trigger default 150,000 input tokens, minimum 50,000; `instructions` replaces the summary prompt; returns a `compaction` block |
| Anthropic | Compaction on demand (beta `compact-2026-09-04`) | you ask for the summary when you choose and put the returned block first in `messages` |
| OpenAI | `context_management=[{"type": "compaction", "compact_threshold": ...}]` on Responses | the server compacts during the response; the compaction item is encrypted, you pass it back as is |
| OpenAI | `client.responses.compact(...)` | stateless: send the window, get a compacted window back |
| Gemini | none found in the docs we checked | do it yourself, as in this lesson |

Here is what the Anthropic request would add, built as a plain dict (not sent: no Anthropic key needed).

In [23]:
anthropic_extra = {
    "betas": ["compact-2026-01-12"],
    "context_management": {"edits": [{
        "type": "compact_20260112",
        "trigger": {"type": "input_tokens", "value": 50_000},     # the smallest allowed
        "instructions": SUMMARY_PROMPT_V2.split("CHAT:")[0].strip(),  # our must-keep rules
    }]},
}
print(json.dumps(anthropic_extra, indent=1)[:500])

{
 "betas": [
  "compact-2026-01-12"
 ],
 "context_management": {
  "edits": [
   {
    "type": "compact_20260112",
    "trigger": {
     "type": "input_tokens",
     "value": 50000
    },
    "instructions": "Summarise this chat between a user and Scout so Scout can continue it without the chat.\nKeep, word for word: the user's goal, every date, and every rule or preference they gave.\nKeep every number Scout gave in an answer, one short line per block of sols (total metres, dustiest sol).\nAls


Two things carry over whichever you use. Their trigger is in the tens of thousands of tokens, not 6,000:
the idea is the same, the scale differs. And the summary prompt decides what survives, so the context rot
test is still your job.

## At work
- **Compaction breaks the prompt cache.** The summary rewrites the start of the request, so the cached
  prefix is lost. Teams compact rarely and in big steps, with a trigger well below the limit. → lesson 41
- **"Scout forgot what I said" is the typical bug report.** Teams keep a plant-and-probe eval (like step 9)
  and rerun it whenever the summary prompt, trigger or model changes. → lesson 48
- **Log every compaction:** tokens before and after, the turns folded in, and the summary text, so a lost
  fact can be traced to the compaction that dropped it. → lesson 49
- **The summary call costs money** and adds latency on the turn that triggers it; Anthropic's on-demand and
  background compaction exist to take it off the user's turn. Count it in the budget. → lesson 54

## What just happened
- Gemini refused a request over its 1,048,576-token window with a 400 `INVALID_ARGUMENT`; nothing was answered.
- Scout got an 8,000-token budget, and the 60-turn chat broke it long before turn 60.
- Clearing old tool results (lesson 31) cut the size a lot but every turn still added tokens.
- `compact` replaced the old turns with a summary and kept the last 6 turns word for word, at a turn boundary.
- With a 6,000-token trigger, all 60 turns stayed under 8,000 tokens.
- The context rot test scored each history against its prompt tokens: full 5/5 (13,226), sliding window
  0/5 (6,507), compacted 4/5 (3,131). It caught a fact the summary dropped; prompt v2 got 5/5 (3,546).

## Common mistakes
**1. Keeping the last N messages instead of the last N turns.** Cutting by message count can split a tool
call from its result. Lesson 31's `check_history` (unchanged) catches the orphan; real APIs reject it too.

In [24]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

Keep the last 22 messages, which sounds like "about 6 turns":

In [25]:
def compact_by_messages(messages: list[dict], summarise, keep: int = 22) -> list[dict]:
    old, recent = messages[:-keep], messages[-keep:]
    return [{"role": "summary", "text": summarise(old), "turn": 0}] + recent

check_history(compact_by_messages(history, extract_summary))

ValueError: tool result for unknown call id 'call_55'

Fix: split at a turn boundary, as `compact` does.

In [26]:
check_history(compact(history, extract_summary))
print("whole turns kept: history is valid")

whole turns kept: history is valid


**2. A summary that keeps the findings but drops the user's rules.** A "just the facts" summariser looks
fine and is small. Run the plant-and-probe check on the summary text itself before shipping it:

In [27]:
def findings_only(messages: list[dict]) -> str:
    return "\n".join(m["text"] for m in messages if m["role"] == "assistant" and "text" in m
                     and m["text"].startswith("Sols"))

summary = compact(history, findings_only)[0]["text"]
for (question, keys) in PROBES[:3]:
    assert any(k in summary.lower() for k in keys), f"summary lost the answer to {question!r}"

AssertionError: summary lost the answer to 'What is my project about?'

Fix: the summary prompt must name what to keep word for word (goal, dates, rules), and the probe check
runs on every summariser change. `extract_summary` passes it:

In [28]:
summary = compact(history, extract_summary)[0]["text"].lower()
print([any(k in summary for k in keys) for _, keys in PROBES])

[True, True, True, True, True]


## Try it
1. Run `chat_with_compaction(60, extract_summary, keep_turns=k)` for `k` in 2, 6 and 12. Print the number
   of compactions and the largest request for each. Which `k` would you choose, and why?
2. Write `rot_check(summary_text) -> float`: the share of `PROBES` whose keywords appear in the summary.
   Use it on the final summary of each run from question 1 (`run["live"][0]["text"]`).

In [29]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [30]:
llm_replay.stop()

## Key terms
- **context overflow error** — the API's refusal of a request whose input is over the model's window, e.g. Gemini's 400 `INVALID_ARGUMENT` "The input token count exceeds the maximum number of tokens allowed".
- **compaction** — replacing the older part of a chat with a short summary, keeping the latest turns word for word, so the history stays under a token budget.
- **summarisation** — asking a model (or code) to rewrite many turns as a short text that keeps what later turns need.
- **compaction trigger** — the token count at which compaction runs, set below the budget so the request that follows still fits.
- **context rot test** — plant facts early in a long chat, ask about them late, and score the answers against the tokens sent; run it whenever context handling changes.